In [1]:
#kc-event-explorer/
#│
#├── WIP- 1.A.py                  
#├── templates/
#│   └── index.html
#│
#├── static/
#│   ├── style.css
#│   └── script.js
#│
#├── data/
#│   └── neighborhoods.geojson
#│
#├── services/
#│   ├── ticketmaster.py
#│   └── geocoder.py
#│
#└── database/
#    └── events.db

In [2]:
#import flask, import API keys from keys.json, import requests
import requests
from flask import Flask, render_template, request
from keys import TICKETMASTER_KEY, EVENTBRITE_TOKEN

app = Flask(__name__)



In [3]:
# Pull events from Ticketmaster
def get_ticketmaster_events():
    url = "https://app.ticketmaster.com/discovery/v2/events.json"

    params = {
       "apikey": TICKETMASTER_KEY,
         "city": "Kansas City",
           "countryCode": "US",
           "size": 50
    } 
    
    response = requests.get(url, params=params)
    return response.json()

In [4]:
# Pull events from Eventbrite
def get_eventbrite_events():
    url = "https://www.eventbriteapi.com/v3/get"

    headers = {
        "Authorization": f"Bearer {EVENTBRITE_TOKEN}"
    }

    params = {
        "location.address": "Kansas City",
        "page_size": 50
    }

    response = requests.get(url, headers=headers, params=params)
    return response.json()

In [5]:
#normalize Ticketmaster data
def normalize_ticketmaster(data):
    events = []
    
    if "_embedded" in data:
        for event in data["_embedded"]["events"]:

            start = event.get("dates", {}).get("start", {})

            raw_time = start.get("localTime")
            time = raw_time[:5] if raw_time else None

            venue = {}

            if "_embedded" in event:
                venues = event["_embedded"].get("venues", [])
                if venues:
                    venue = venues[0]

            # GET CATEGORY
            classification = event.get("classifications", [])

            category = "Other"

            if classification:
                category = classification[0].get(
                    "segment", {}
                ).get(
                    "name",
                    "Other"
                )

            # NORMALIZE CATEGORY NAMES
            if category == "Arts & Theatre":
                category = "Theater"

            elif category not in [
                "Music",
                "Sports",
                "Theater",
                "Comedy"
            ]:
                category = "Other"
        

            events.append({
                "name": event.get("name"),
                "date": start.get("localDate"),
                "time": time,
                "source": "ticketmaster",
                "lat": venue.get("location", {}).get("latitude"),
                "lon": venue.get("location", {}).get("longitude"),
                "city": venue.get("city", {}).get("name"),
                "class": event["classifications"][0]["segment"]["name"]
            })

    return events
    

In [6]:
def get_eventbrite_events():

    url = "https://www.eventbriteapi.com/v3/get"


    headers = {
        "Authorization": f"Bearer {EVENTBRITE_TOKEN}"
    }

    params = {
        "location.address": "Kansas City",
        "page_size": 50
    }

    response = requests.get(url, headers=headers, params=params)

    print("EVENTBRITE STATUS:", response.status_code)

    data = response.json()

    print("EVENTBRITE EVENTS:", len(data.get("events", [])))

    print("EVENTBRITE RESPONSE:", response.text[:500])

    return data

In [7]:
#normalize Eventbrite data
def normalize_eventbrite(data):
    events = []

    for event in data.get("events", []):

        name = event.get("name", {}).get("text")
        start = event.get("start", {}).get("local")

        date = None
        time = None
        city = event.get("venue", {}).get("address", {}).get("city")

        if start:
            date = start[:10]
            time = start[11:16]

        events.append({
            "name": name,
            "date": date,
            "time": time,
            "source": "eventbrite",
            "lat": None,
            "lon": None,
            "city": city 
        })

    return events

from datetime import datetime


In [8]:
def get_all_events():

    ticketmaster_data = get_ticketmaster_events()
    eventbrite_data = get_eventbrite_events()

    ticketmaster_events = normalize_ticketmaster(ticketmaster_data)
    eventbrite_events = normalize_eventbrite(eventbrite_data)
    return ticketmaster_events + eventbrite_events


In [9]:
from asyncio import events


@app.route("/")
def home():

    print("ROUTE HIT")

    selected_date = request.args.get("date")
    selected_time = request.args.get("time")
    selected_category = request.args.get("category")

    data = get_all_events()

    # Apply remaining filters
    filtered = data

    from datetime import datetime

    filtered = data

    # TIME FILTER
    if selected_time:
        try:
            selected_time_obj = datetime.strptime(selected_time, "%H:%M").time()

            filtered = [
                e for e in filtered
                if e.get("time")
                and datetime.strptime(e["time"], "%H:%M").time() >= selected_time_obj
            ]

        except:
            filtered = filtered


    # DATE FILTER
    if selected_date:
        try:
            filtered = [
                e for e in filtered
                if e.get("date") == selected_date
            ]
        except:
            filtered = filtered

    # Build final display list
    events = []

    for event in filtered:

        display_time = "TBD"

        if event.get("time"):
            try:
                display_time = datetime.strptime(
                    event["time"],
                    "%H:%M"
                ).strftime("%I:%M %p")
            except:
                display_time = event["time"]

        if selected_category:
            filtered = [
                e for e in filtered
                if e.get("category") == selected_category]

        events.append({
            "name": event.get("name"),
            "date": event.get("date"),
            "time": display_time,
            "source": event.get("source"),
            "lat": event.get("lat"),
            "lon": event.get("lon"),
            "classification": event.get("class"),
        })

    print("EVENT COUNT:", len(events))
    
    return render_template(
        "index.html",
        events=events,
        selected_date=selected_date,
        selected_time=selected_time
    )

In [10]:
#run Flask app
if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5001, debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


ROUTE HIT


127.0.0.1 - - [10/Jul/2026 14:16:25] "GET / HTTP/1.1" 200 -


EVENTBRITE STATUS: 404
EVENTBRITE EVENTS: 0
EVENTBRITE RESPONSE: {"status_code":404,"error_description":"The path you requested does not exist.","error":"NOT_FOUND"}
EVENT COUNT: 50
ROUTE HIT


...

EVENTBRITE STATUS: 404
EVENTBRITE EVENTS: 0
EVENTBRITE RESPONSE: {"status_code":404,"error_description":"The path you requested does not exist.","error":"NOT_FOUND"}
EVENT COUNT: 1
ROUTE HIT


127.0.0.1 - - [10/Jul/2026 14:22:23] "GET /?date=2026-07-11&time=14:18&category= HTTP/1.1" 200 -


EVENTBRITE STATUS: 404
EVENTBRITE EVENTS: 0
EVENTBRITE RESPONSE: {"status_code":404,"error_description":"The path you requested does not exist.","error":"NOT_FOUND"}
EVENT COUNT: 1
ROUTE HIT


127.0.0.1 - - [10/Jul/2026 14:22:27] "GET /?date=2026-07-11&time=14:18&category=Comedy HTTP/1.1" 200 -


EVENTBRITE STATUS: 404
EVENTBRITE EVENTS: 0
EVENTBRITE RESPONSE: {"status_code":404,"error_description":"The path you requested does not exist.","error":"NOT_FOUND"}
EVENT COUNT: 1
